# How a portfolio works

A `qs.Portfolio` holds a list of positions and adds up their valuation
results. This notebook builds a small USD book and values it. We then check
its PV, risk and payments against the individual positions. Finally, we show
what happens with repeated trades, opposite trades and different requests.

The book contains a deposit, a swap in which we receive fixed, and a smaller
position in the same swap in which we pay fixed. The market is
**illustrative, as of 24 February 2026**. All market inputs are defined in the
notebook, so no data files or downloads are needed. Run the cells in order
with the package built from this checkout and
[the example dependencies](../README.md#install-and-run).


In [1]:
import pandas as pd
import quantsupport as qs
from IPython.display import display

VALUATION_DATE = "2026-02-24"
START = "2026-03-02"
REQUESTS = [qs.Request.Value, qs.Request.Sensitivities, qs.Request.Cashflows()]


## 1. Instruments describe contracts; trades describe positions

An instrument holds the contract terms. The deposit has a principal, a coupon
and a maturity. The swap has a fixed leg and a floating leg. A `Trade` turns
an instrument into a position by adding a trade date, a side and, optionally,
a size:

- A `LongReceive` deposit pays out the principal, then receives interest and
  the repayment.
- A `LongReceive` swap receives fixed and pays floating.
- A `PayShort` swap pays fixed and receives floating.

Use the side to reverse a position. Use a non-negative `notional` on the trade
to set its size.

Both swap positions below use the same instrument. The second trade has a
notional of 500,000; the instrument's own notional stays at 1,000,000. We keep
our own labels for the positions, because two positions can share an
instrument identifier.


In [2]:
deposit = qs.FixedRateDeposit(
    identifier="USD-DEPOSIT", start_date=START, maturity_date="2027-03-02",
    notional=100_000, rate=0.04, currency="USD", discount_index="SOFR",
)
swap = qs.Swap(
    identifier="USD-SWAP", start_date=START, maturity_date="2028-03-02",
    notional=1_000_000, fixed_rate=0.04, currency="USD", market_index="SOFR",
    fixed_leg_frequency="Semiannual", floating_leg_frequency="Quarterly",
)
receive_fixed = qs.Trade(swap, VALUATION_DATE, qs.Side.LongReceive)
pay_half = qs.Trade(swap, VALUATION_DATE, qs.Side.PayShort, notional=500_000)
positions = {
    "Deposit": qs.Trade(deposit, VALUATION_DATE, qs.Side.LongReceive),
    "Receive fixed": receive_fixed,
    "Pay fixed, half size": pay_half,
}
portfolio = qs.Portfolio("USD-BOOK", list(positions.values()))

print(portfolio)
pd.DataFrame([
    {
        "position": label,
        "instrument": trade.instrument.identifier,
        "side": str(trade.side),
        "notional_usd": (
            trade.notional if trade.notional is not None else trade.instrument.notional
        ),
    }
    for label, trade in positions.items()
])


Portfolio(identifier="USD-BOOK", trades=3)


,position,instrument,side,notional_usd
0,Deposit,USD-DEPOSIT,LongReceive,100000.0
1,Receive fixed,USD-SWAP,LongReceive,1000000.0
2,"Pay fixed, half size",USD-SWAP,PayShort,500000.0


## 2. Give every member a common pricing context

The portfolio holds only the trades. The `PricingContext` supplies the market
data, discounting rules and pricers. Here one SOFR curve projects the
floating payments and discounts all USD payments. The curve is defined by
continuously compounded zero rates, given as annual decimals: `0.03` is 3%.
All accrual periods start after the valuation date, so no past fixings are
needed.

The helper creates a new context for each calculation. Inside
`with context:`, the context checks its inputs and builds the curves that
each calculation needs. When the block ends, it releases them. Use one
context at a time. The returned `EvaluationResults` can be used after the
block.


In [3]:
PILLARS = ["0D", "1Y", "5Y"]
ZERO_RATES = [0.03, 0.032, 0.035]


def make_context(rate_shift=0.0, include_eur=False):
    inputs = [("USD", "SOFR", PILLARS, [rate + rate_shift for rate in ZERO_RATES])]
    if include_eur:
        inputs.append(("EUR", "ESTR", ["0D", "5Y"], [0.03, 0.03]))
    quotes = qs.QuoteStore.from_dict({
        "reference_date": VALUATION_DATE,
        "quotes": [
            {"identifier": f"ZeroRate_{currency}_{index}_{pillar}", "mid": rate}
            for currency, index, pillars, rates in inputs
            for pillar, rate in zip(pillars, rates)
        ],
    })
    curves = [qs.CurveConfiguration(
        market_index=index, day_counter="Actual360", interpolator="Linear",
        quotes=[f"ZeroRate_{currency}_{index}_{pillar}" for pillar in pillars],
    ) for currency, index, pillars, _ in inputs]
    return qs.PricingContext(
        quotes=quotes, curves=curves,
        csa_policy=qs.SingleCurveCSADiscountPolicy("USD", "SOFR"),
    )


pd.DataFrame({"pillar": PILLARS, "zero_rate": ZERO_RATES})


,pillar,zero_rate
0,0D,0.030
1,1Y,0.032
2,5Y,0.035


## 3. Evaluate the portfolio

`context.evaluate(portfolio, requests)` returns the same `EvaluationResults`
type as for a single trade. Without `requests`, it computes only `Value`. Here
we also ask for sensitivities and cashflows.

PVs and sensitivities are reported in the context's base currency, which
the collateral policy sets to USD here. Positions in other currencies are
converted at spot during valuation, so their sensitivities include the
conversion. Cashflows stay in their own payment currencies.

The portfolio values each position with that position's own pricer and adds
the results:

| Step | What happens |
| --- | --- |
| Read a position | Keep the list order, including repeated entries |
| Price the position | Use the pricer for that position's product, with the same context |
| Combine results | Add the PVs, add sensitivities with the same label, and combine cashflows |
| Return | One result labeled with the portfolio identifier, here `USD-BOOK` |

The result holds only the totals, not the individual position results. Below
we also value each position separately, only to explain and check the total.


In [4]:
with make_context() as context:
    value_only = context.evaluate(portfolio)
    result = context.evaluate(portfolio, REQUESTS)
    individual = {
        label: context.evaluate(trade, REQUESTS)
        for label, trade in positions.items()
    }

pd.DataFrame([{
    "identifier": result.identifier,
    "evaluation_date": str(result.evaluation_date),
    "positions": len(portfolio),
    "pv_usd": result.price,
}])


,identifier,evaluation_date,positions,pv_usd
0,USD-BOOK,2026-02-24,3,7506.367672


## 4. Reconcile PV and position sizes

The portfolio PV is the sum of the position PVs, each with its sign, in the
common reporting currency:

$$V_{\mathrm{portfolio}} = \sum_i V_i.$$

The half-size pay-fixed position is worth minus half the receive-fixed
position. Each trade's PV includes its side and size, so do not apply them
again. The aggregate result does not carry our position labels; they are
only in our separate table.


In [5]:
position_values = pd.DataFrame([
    {"position": label, "pv_usd": item.price}
    for label, item in individual.items()
]).set_index("position")
display(position_values)
print(f"Portfolio PV: USD {result.price:,.2f}")


,pv_usd
position,
Deposit,728.763613
Receive fixed,13555.208117
"Pay fixed, half size",-6777.604058


Portfolio PV: USD 7,506.37


## 5. Add sensitivities by their input label

For a market input $q_k$ shared by several positions, the portfolio
sensitivity is the sum of the position sensitivities:
$\partial V_{\mathrm{portfolio}}/\partial q_k =
\sum_i \partial V_i/\partial q_k$. Match rows by their `pillar` label, not by
row order.

`result.sensitivities` has `pillar` and `value` columns. Each `value` is the
change in PV per unit change in the input. Here the inputs are zero rates in
decimals, so multiplying by `1e-4` gives the USD P&L for a +1 bp move. The P&L
for a parallel +1 bp move of all rates is the sum of these scaled values.
Volatility, FX and spread inputs have different units and need their own
scaling.


In [6]:
trade_risk = pd.concat([
    item.sensitivities.assign(position=label)
    for label, item in individual.items()
], ignore_index=True)
expected_risk = trade_risk.groupby("pillar")["value"].sum().sort_index()
actual_risk = result.sensitivities.set_index("pillar")["value"].sort_index()

risk_report = actual_risk.rename("dPV_dZeroRate").to_frame()
risk_report["pnl_plus_1bp_usd"] = actual_risk * 1e-4
risk_report


,dPV_dZeroRate,pnl_plus_1bp_usd
pillar,,
ZeroRate_USD_SOFR_0D,7286.052758,0.728605
ZeroRate_USD_SOFR_1Y,-856863.863478,-85.686386
ZeroRate_USD_SOFR_5Y,-251069.066639,-25.106907


### Check risk with full revaluation

Here we move all three zero rates by +1 bp and by −1 bp, each in a new
context, and value the book again. The central difference
$[V(q+h)-V(q-h)]/(2h)$ should match the sum of the sensitivities. The P&L for
the +1 bp move alone differs slightly from the linear estimate because PV is
not linear in rates.


In [7]:
bump = 1e-4
with make_context(bump) as context:
    plus = context.evaluate(portfolio).price
with make_context(-bump) as context:
    minus = context.evaluate(portfolio).price

central_derivative = (plus - minus) / (2 * bump)
aad_derivative = actual_risk.sum()
pd.DataFrame([{
    "summed_derivative_usd_per_unit": aad_derivative,
    "central_difference_usd_per_unit": central_derivative,
    "linear_pnl_plus_1bp_usd": aad_derivative * bump,
    "revalued_pnl_plus_1bp_usd": plus - result.price,
}])


,summed_derivative_usd_per_unit,central_difference_usd_per_unit,linear_pnl_plus_1bp_usd,revalued_pnl_plus_1bp_usd
0,-1.100647e+06,-1.100647e+06,-110.064688,-110.053977


## 6. Read cashflows and payment totals

The result has two pandas tables of payments:

| Property | Contents |
| --- | --- |
| `cashflows` | Each payment, signed and undiscounted, with its date, type, currency and leg. It also shows contract terms such as signed notional, accrual dates, rate, spread and fixing index |
| `cashflow_summary` | Net `interest`, `capital`, `other` and `total` by payment date and currency |

Coupons with matching terms may be merged into one row. When fixed rates or
floating spreads differ, the merged row shows their notional-weighted
average. Coupons with different types, dates, conventions or fixing indices
stay in separate rows. Payments in opposite directions also stay in separate
rows, even though the summary nets them. The summary shows cash paid on each
payment date; it does not spread interest income over accrual periods.

Below, each swap coupon date has a row for the full-size receive-fixed
position and a row for the half-size opposite position. Receipts are positive
and payments are negative.


In [8]:
cashflows = result.cashflows
coupon_rows = cashflows["type"].isin(["FixedRateCoupon", "FloatingRateCoupon"])
cashflows.loc[coupon_rows, [
    "payment_date", "type", "accrual_start", "accrual_end", "market_index",
    "notional", "rate", "spread", "amount", "currency",
]].head(8)


,payment_date,type,accrual_start,accrual_end,market_index,notional,rate,spread,amount,currency
4,2026-06-02,FloatingRateCoupon,2026-03-02,2026-06-02,SOFR,-1000000.0,NaN,0.0,-7842.893950,USD
5,2026-06-02,FloatingRateCoupon,2026-03-02,2026-06-02,SOFR,500000.0,NaN,0.0,3921.446975,USD
6,2026-09-02,FixedRateCoupon,2026-03-02,2026-09-02,NaN,1000000.0,0.04,NaN,20444.444444,USD
7,2026-09-02,FixedRateCoupon,2026-03-02,2026-09-02,NaN,-500000.0,0.04,NaN,-10222.222222,USD
8,2026-09-02,FloatingRateCoupon,2026-06-02,2026-09-02,SOFR,-1000000.0,NaN,0.0,-8102.604188,USD
9,2026-09-02,FloatingRateCoupon,2026-06-02,2026-09-02,SOFR,500000.0,NaN,0.0,4051.302094,USD
10,2026-12-02,FloatingRateCoupon,2026-09-02,2026-12-02,SOFR,-1000000.0,NaN,0.0,-8269.714496,USD
11,2026-12-02,FloatingRateCoupon,2026-09-02,2026-12-02,SOFR,500000.0,NaN,0.0,4134.857248,USD


### Reconcile payments and reconstruct PV

First we check that the net payments equal the sum of the individual trades'
payments. Then we discount them on the same SOFR curve the pricers use. This
gives back the PV for this book of a deposit and swaps, whose payments
are known once the curve is fixed. Products with options or credit risk would
need more than discounting.

The summary includes the initial principal payments. The deposit pays out
−100,000 of capital on 2 March 2026 and receives +100,000 back at maturity.
The swaps' principal exchanges cancel in the net summary. Adding up the
undiscounted totals does not give the PV.


In [9]:
payments = pd.concat([item.cashflows for item in individual.values()], ignore_index=True)
expected_payments = payments.groupby(["payment_date", "currency"])["amount"].sum()
summary = result.cashflow_summary
actual_payments = summary.set_index(["payment_date", "currency"])["total"]

with make_context() as context:
    curve = context.curve(qs.MarketIndex.SOFR)
    payment_values = summary.assign(
        discount_factor=summary["payment_date"].map(curve.discount_factor)
    )
payment_values["pv_usd"] = payment_values["total"] * payment_values["discount_factor"]
print(f"Discounted payments: USD {payment_values['pv_usd'].sum():,.2f}")
payment_values


Discounted payments: USD 7,506.37


,payment_date,currency,interest,capital,other,total,discount_factor,pv_usd
0,2026-03-02,USD,0.000000,-100000.0,0.0,-100000.000000,0.999500,-99949.957731
1,2026-06-02,USD,-3921.446975,0.0,0.0,-3921.446975,0.991722,-3888.983707
2,2026-09-02,USD,6170.920128,0.0,0.0,6170.920128,0.983751,6070.646796
3,2026-12-02,USD,-4134.857248,0.0,0.0,-4134.857248,0.975682,-4034.306019
4,2027-03-02,USD,9907.556560,100000.0,0.0,109907.556560,0.967548,106340.811900
5,2027-06-02,USD,-4229.734265,0.0,0.0,-4229.734265,0.959431,-4058.140271
6,2027-09-02,USD,5943.799770,0.0,0.0,5943.799770,0.951291,5654.285854
7,2027-12-02,USD,-4279.619008,0.0,0.0,-4279.619008,0.943218,-4036.614592
8,2028-03-02,USD,5783.851939,0.0,0.0,5783.851939,0.935125,5408.625441


## 7. Membership preserves repeated positions

A portfolio is a non-empty ordered list. Duplicates are kept: listing the
same trade twice doubles its exposure. An equal-size opposite trade cancels
the PV completely. Our original half-size opposite trade cancels only half.

To offset positions, add opposite trades. A portfolio has no weights
parameter. For legal netting with a counterparty, collateral and future
exposure, use `NettingSet` with its CSA terms.


In [10]:
pay_full = qs.Trade(swap, VALUATION_DATE, qs.Side.PayShort)
doubled = qs.Portfolio("PORTFOLIO", [receive_fixed, receive_fixed])
offset = qs.Portfolio("PORTFOLIO", [receive_fixed, pay_full])
with make_context() as context:
    doubled_value = context.evaluate(doubled).price
    offset_value = context.evaluate(offset).price

single_value = individual["Receive fixed"].price
pd.DataFrame([
    {"membership": "receive", "positions": 1, "pv_usd": single_value},
    {"membership": "receive, receive", "positions": len(doubled), "pv_usd": doubled_value},
    {"membership": "receive, equal-size pay", "positions": len(offset), "pv_usd": offset_value},
])


,membership,positions,pv_usd
0,receive,1,13555.208117
1,"receive, receive",2,27110.416234
2,"receive, equal-size pay",2,0.000000


## 8. Request only measures that can be aggregated

A portfolio adds up only `Value`, `Sensitivities` and `Cashflows`. Ask for
trade-specific results such as `FairRate`, yield or duration on individual
trades. A portfolio ignores other requests. An empty request list computes
nothing, while omitting `requests` computes the PV.

A measure that was not requested is `None`. If any position cannot supply a
requested measure, that measure is `None` for the whole portfolio, not zero
or a partial total. If pricing fails for any position, the evaluation raises
an error and returns no result. `Portfolio("EMPTY", [])` raises an error
when you create it.

The same `context.evaluate` call handles PV, risk and cashflow requests. The
cell below checks these rules on the same book.


In [11]:
with make_context() as context:
    flows_only = context.evaluate(portfolio, [qs.Request.Cashflows()])
    with_fair_rate = context.evaluate(
        portfolio, [qs.Request.Value, qs.Request.FairRate(0)],
    )
    no_measures = context.evaluate(portfolio, [])

pd.DataFrame([
    {
        "requests": label,
        "has_pv": item.price is not None,
        "has_sensitivities": item.sensitivities is not None,
        "has_cashflows": item.cashflows is not None,
        "has_fair_rate": item.fair_rate is not None,
    }
    for label, item in [
        ("default (Value)", value_only),
        ("Value, Sensitivities, Cashflows", result),
        ("Cashflows", flows_only),
        ("Value, FairRate(0)", with_fair_rate),
        ("[]", no_measures),
    ]
])


,requests,has_pv,has_sensitivities,has_cashflows,has_fair_rate
0,default (Value),True,False,False,False
1,"Value, Sensitivities, Cashflows",True,True,True,False
2,Cashflows,False,False,True,False
3,"Value, FairRate(0)",True,False,False,False
4,[],False,False,False,False


## 9. Positions in other currencies are converted

Now we add a EUR deposit, an ESTR curve and a spot rate of
**1.10 USD per EUR**. The same `context.evaluate` call returns the deposit's
PV and sensitivities in USD. A portfolio that holds it alongside the USD
positions needs no currency argument. The payment summary keeps EUR and USD
payments apart.

Discounting the deposit's EUR payments on the ESTR curve gives its PV in EUR.
Multiplying that by spot checks the USD result. The sensitivity to the
`EUR/USD` rate equals the EUR PV.


In [12]:
eur_deposit = qs.Trade(qs.FixedRateDeposit(
    identifier="EUR-DEPOSIT", start_date=START, maturity_date="2027-03-02",
    notional=100_000, rate=0.04, currency="EUR", discount_index="ESTR",
), VALUATION_DATE)
mixed_portfolio = qs.Portfolio("PORTFOLIO", [*positions.values(), eur_deposit])
context = make_context(include_eur=True)
fx = qs.FxStore()
fx.add("EUR", "USD", 1.10)
context.fx = fx

with context:
    foreign = context.evaluate(eur_deposit, REQUESTS)
    mixed = context.evaluate(mixed_portfolio, REQUESTS)
    eur_curve = context.curve("ESTR")
    local_pv_eur = sum(
        payment.amount * eur_curve.discount_factor(payment.payment_date)
        for payment in foreign.cashflows.itertuples(index=False)
    )

pd.DataFrame([
    {"position": "Original USD book", "pv_usd": result.price},
    {"position": "EUR deposit (converted automatically)", "pv_usd": foreign.price},
    {"position": "Mixed portfolio", "pv_usd": mixed.price},
])


,position,pv_usd
0,Original USD book,7506.367672
1,EUR deposit (converted automatically),1031.485198
2,Mixed portfolio,8537.852870


For merged coupons, separate fixing indices and CSV exports, continue with
[coupon and cashflow aggregation](portfolio_aggregation.ipynb). For a book of
swaps, bonds and caplets with separate rate and volatility scenarios, read
[portfolio risk](portfolio_risk.ipynb). For counterparty agreements and future
exposure, see [netting and XVA](../cva/xva.ipynb).

The tables are ordinary pandas objects. For example,
`result.sensitivities.to_csv("portfolio_risk.csv", index=False)` exports the
sensitivities, and `result.cashflow_summary.to_csv("payments.csv", index=False)`
exports the undiscounted payment totals. Record the input units and
currencies with the exported files.